# Hangman letter analysis

Letter-focused tables split out from `strategy_explorer.ipynb`: where a letter
gets guessed, what a set of guessed letters resolves to, letter frequency by
position, and common n-grams. Same `strategy.py` / `strategy_view.py` backend and
the `LENGTH` / `WEIGHTING` / `STRATEGY` knobs below (each table also takes explicit
args, so you can override per call).

In [1]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'strategy.py').exists():
            return c
    raise FileNotFoundError('Could not find strategy.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import strategy
import strategy_view as sview
import wordlist

pd.set_option('display.max_rows', 200)   # show up to 200 rows before pandas truncates to a head/tail preview

print(f'imported strategy + strategy_view from {math_dir}')

imported strategy + strategy_view from /Users/patrickliscio/My Documents/Pycharm Projects/Ballpark-Figures/hangman/math


## Defaults

These knobs are the defaults the tables below use. `WEIGHTING` sets the frequency
weighting for **every** table. `STRATEGY` / `BUDGET` / `FILL_CAPPED` affect only the
strategy-dependent tables (`guesses_of_letter*`, `letters_summary*`); the position
and n-gram tables are **wordlist-only**. `LENGTH` is the default length for the
single-length tables. Tables that sweep every length build their own per-length
views, so unsolved `optimal`/`winprob` lengths are skipped cleanly.

In [2]:
LENGTH    = 5           # <-- default word length for single-length tables
WEIGHTING = "uniform"   # <-- uniform, zipf, greedy, best_opener, greedy_word, best_opener_word
STRATEGY  = "optimal"   # <-- greedy, greedy_best, optimal, winprob (strategy-dependent tables only)
BUDGET    = None        # <-- winprob only: miss budget k (None = most permissive solved)
FILL_CAPPED = True      # <-- optimal only: finish cap-truncated branches with greedy

print(f'defaults -> length {LENGTH}, weighting {WEIGHTING!r}, strategy {STRATEGY!r}')

defaults -> length 5, weighting 'uniform', strategy 'optimal'


## Where letters are guessed (single length)

`guesses_of_letter(letters, length)` finds **every** situation -- a reachable board
with **nonzero probability** under the weighting -- at `length` where the strategy's
next guess is one of `letters` (a string like `'u'` or `'aeiou'`), sorted by **level**
(depth = guesses already made) then by **freq_words** (descending). Columns:

- **level** -- guesses already made when this board is reached (0 = opener),
- **letter** -- which of `letters` the strategy guesses there,
- **board** / **missed** -- the revealed board and the missed letters there,
- **n_words** -- words consistent with that board,
- **freq_words** -- frequency-weighted words (`fmass / total_freq_L * N_L`); for a
  single length this is just a scaling of the raw frequency mass,
- **sample** -- a few example candidate words at that board,
- **hit_sample** -- a few candidates that actually **contain that letter** -- the
  words the guess would reveal (a hit).

Boards whose words all weigh 0 under the weighting are skipped (nonzero-prob only).
With several letters, rows interleave by level/freq -- sort by `letter` to group them.
Uses the current `STRATEGY`/`WEIGHTING`/`BUDGET`/`FILL_CAPPED`.

In [ ]:
LETTERS  = 'u'       # <-- one or more letters to locate (e.g. 'u' or 'aeiou')
WORD_LEN = LENGTH    # <-- word length (defaults to the LENGTH picked at the top)

def guesses_of_letter(letters, length, *, strategy=STRATEGY, weighting=WEIGHTING,
                      budget=BUDGET, fill_capped=FILL_CAPPED, max_level=40,
                      verbose=True):
    """Every situation at `length` where the strategy's next guess is one of `letters`
    (a string like 'u' or 'aeiou') and the board has nonzero probability under the
    weighting, sorted by level (depth) then freq_words (descending). Columns: level,
    letter (which letter is guessed there), board, missed, n_words, freq_words
    (= fmass / total_freq_L * N_L), sample (any candidates there), hit_sample
    (candidates that CONTAIN that letter -- the words the guess would reveal).
    Determined branches (no guess) and zero-frequency boards are skipped.
    `verbose=False` silences the skip/empty notes (used by the all-lengths sweep)."""
    letterset = {c for c in letters.lower() if c.isalpha()}
    cols = ['level', 'letter', 'board', 'missed', 'n_words', 'freq_words', 'sample', 'hit_sample']
    s = sview.make_strategy(strategy, length, weighting=weighting, budget=budget,
                            fill_capped=fill_capped)
    if not s.available:
        if verbose:
            print('not available:', s.note)
        return pd.DataFrame(columns=cols)
    total = float(s.word_freq().sum())
    scale = (s.g.N / total) if total else 0.0          # freq share -> effective words
    rows = []
    for level in range(max_level + 1):
        nodes = s.states_at_depth(level)
        if not nodes:                                   # tree bottomed out
            break
        for st in nodes:
            if st['determined'] or st['guess'] not in letterset or st['fmass'] == 0.0:
                continue
            g_letter = st['guess']
            missed = ''.join(st['missed'])
            cand = s.state(st['board'], missed).get('candidates', [])
            hit = [w for w in cand if g_letter in w]     # words this guess reveals
            rows.append({'level': level, 'letter': g_letter, 'board': st['board'],
                         'missed': missed, 'n_words': st['n'],
                         'freq_words': st['fmass'] * scale,
                         'sample': ' '.join(st['sample'][:5]),
                         'hit_sample': ' '.join(hit[:5])})
    if not rows:
        if verbose:
            shown = ''.join(sorted(letterset))
            print(f'{shown!r} is never the next guess at a nonzero-probability board '
                  f'(length {length}, {s.label})')
        return pd.DataFrame(columns=cols)
    return (pd.DataFrame(rows)
              .sort_values(['level', 'freq_words'], ascending=[True, False])
              .reset_index(drop=True))

res = guesses_of_letter(LETTERS, WORD_LEN)
shown = ''.join(sorted({c for c in LETTERS.lower() if c.isalpha()}))
print(f'{shown!r} at length {WORD_LEN}: {len(res)} situations  [{STRATEGY} / {WEIGHTING}]')
with pd.option_context('display.max_colwidth', None):
    display(res)

## Where letters are guessed (all lengths)

`guesses_of_letter_all_lengths(letters)` is the same as above but swept over **every
word length**: every situation (at any length) where the strategy's next guess is one
of `letters`, sorted by **level**, then **length**, then **freq_words** (descending).
It adds a `length` column; because `freq_words` is the cross-length-comparable metric
(`fmass / total_freq_L * N_L`), situations from different lengths sit on the same scale
and sort together. The `letter` column shows which letter is guessed. Lengths with no
committed solve are skipped. Uses the current `STRATEGY`/`WEIGHTING`/`BUDGET`/`FILL_CAPPED`.

In [ ]:
def guesses_of_letter_all_lengths(letters, lengths=range(1, 21), *,
                                  strategy=STRATEGY, weighting=WEIGHTING,
                                  budget=BUDGET, fill_capped=FILL_CAPPED):
    """`guesses_of_letter` swept over every word length: every situation (any length)
    where the strategy's next guess is one of `letters` and the board has nonzero
    probability, sorted by level, then length, then freq_words (descending). Adds a
    `length` column; `freq_words` is the cross-length-comparable metric
    (fmass / total_freq_L * N_L), so situations from different lengths sort together.
    The `letter` column shows which letter is guessed."""
    cols = ['level', 'length', 'letter', 'board', 'missed', 'n_words', 'freq_words',
            'sample', 'hit_sample']
    parts = []
    for L in lengths:
        df = guesses_of_letter(letters, L, strategy=strategy, weighting=weighting,
                               budget=budget, fill_capped=fill_capped, verbose=False)
        if len(df):
            df.insert(1, 'length', L)
            parts.append(df)
    if not parts:
        shown = ''.join(sorted({c for c in letters.lower() if c.isalpha()}))
        print(f'{shown!r} is never guessed at any length  [{strategy} / {weighting}]')
        return pd.DataFrame(columns=cols)
    return (pd.concat(parts, ignore_index=True)
              .sort_values(['level', 'length', 'freq_words'],
                           ascending=[True, True, False])
              .reset_index(drop=True))

res_all = guesses_of_letter_all_lengths(LETTERS)
shown = ''.join(sorted({c for c in LETTERS.lower() if c.isalpha()}))
print(f'{shown!r}, all lengths: {len(res_all)} situations  [{STRATEGY} / {WEIGHTING}]')
with pd.option_context('display.max_colwidth', None):
    display(res_all.head(200))

## Summary for a set of guessed letters

`letters_summary(letters)` takes a collection of `letters` (a string; case, dupes
and non-letters are ignored) treated as **already guessed**, and reports over the
current `LENGTH`'s word list a word **count** and **total frequency** (the
weighting's `fmass`, and its % of all words/frequency) for these rows:

- **only these letters** — words spelled entirely from `letters` (so fully revealed
  once all are guessed);
- **uniquely determined after** — words whose revealed pattern of `letters` (which of
  them sit at which positions, blanks elsewhere) is shared by no other word, so
  guessing `letters` already pins the word down;
- **misses after = k** — one row per number of **further misses** the strategy makes
  finishing the word once `letters` have all been guessed (`k = 0` is the miss-free
  case). A trailing **unsolved** row appears if any word can't be solved at all.

The `misses after = k` rows **partition** every solvable word; the first two rows are
(overlapping) subsets of `misses after = 0` (only-these ⊆ uniquely-determined ⊆
miss-free).

> **Miss-count semantics (force-guessed).** We **force-guess** all of `letters` — reaching
> the board where each is revealed (or known absent; absent ones are misses *during*
> that phase, which don't count) — then **finish greedily** and count the further misses
> (a miss still narrows: the letter is now known absent). For the `greedy`/`greedy_best`
> strategies this greedy finish IS the strategy; for `optimal`/`winprob` the continuation
> from an arbitrary forced state isn't in the committed tree, so the finish is greedy
> (greedy-below-`L`, the same convention as `FILL_CAPPED`). Say if you'd rather it use the
> selected strategy's own play order instead of a forced-`L`-first model.

In [ ]:
GUESSED_LETTERS = 'aeiounslrt'   # <-- letters treated as already guessed (case/dupes/non-letters ignored)

def _greedy_misses_after(g, freq, wi, Lidx, guessed0):
    """Number of misses a GREEDY play makes finishing word `wi` after force-guessing
    the letters `Lidx` (bitmask `guessed0`) -- how many of greedy's remaining guesses
    are absent from `wi`. 0 == solved miss-free from the L-state."""
    m = g.masks
    S = np.arange(g.N)
    for c in Lidx:                                  # candidates sharing wi's L-reveal
        S = S[m[S, c] == m[wi, c]]
    guessed, misses = guessed0, 0
    while S.shape[0] > 1:
        lf = freq[S] @ g.present[S]                 # greedy score: freq-weighted presence
        gg = guessed
        while gg:                                   # mask already-guessed letters
            c0 = (gg & -gg).bit_length() - 1
            lf[c0] = -1.0
            gg &= gg - 1
        c = int(lf.argmax())
        if lf[c] < 0:
            break
        if not m[wi, c]:                            # greedy guesses a letter absent from wi -> miss
            misses += 1
        S = S[m[S, c] == m[wi, c]]                  # a miss still narrows (letter known absent)
        guessed |= (1 << c)
    return misses

def _letter_stats(s, letters):
    """Per-word arrays for guessed `letters` (aligned to `s.g.words`): only_L (spelled
    only from them), uniq (uniquely determined once they're guessed), misses_after
    (# further misses the greedy finish makes after force-guessing them; -1 for words
    the strategy can't solve at all)."""
    g, freq, words = s.g, s.word_freq(), s.g.words
    Lset = {c for c in letters.lower() if c.isalpha()}
    Lidx = {ord(c) - 97 for c in Lset}
    guessed0 = 0
    for c in Lidx:
        guessed0 |= (1 << c)
    outside = [c for c in range(26) if c not in Lidx]
    only_L = (g.present[:, outside].sum(axis=1) == 0) if Lidx else np.zeros(g.N, bool)
    sigs = [''.join(ch if ch in Lset else '_' for ch in w) for w in words]
    counts = pd.Series(sigs).value_counts()
    uniq = np.array([counts[si] == 1 for si in sigs])
    status = s.per_word_status()                    # 1 capped / 2 lost -> unsolved
    misses_after = np.array([-1 if status[i] in (1, 2)
                             else _greedy_misses_after(g, freq, i, Lidx, guessed0)
                             for i in range(g.N)], dtype=int)
    return only_L, uniq, misses_after

def _letter_rows(only_L, uniq, misses_after):
    """Ordered (label, boolean-mask) rows: the two descriptive categories, then one
    row per # of further misses (0, 1, ... up to the max seen), then unsolved if any.
    The `misses after = k` rows PARTITION the solvable words; the first two are
    (overlapping) subsets of `misses after = 0`."""
    solved = misses_after >= 0
    rows = [('only these letters', only_L), ('uniquely determined after', uniq)]
    kmax = int(misses_after[solved].max()) if solved.any() else -1
    for k in range(kmax + 1):
        rows.append((f'misses after = {k}', misses_after == k))
    if (~solved).any():
        rows.append(('unsolved (capped/lost)', ~solved))
    return rows

def letters_summary(letters, *, strategy=STRATEGY, length=LENGTH, weighting=WEIGHTING,
                    budget=BUDGET, fill_capped=FILL_CAPPED):
    """Per-category count + total frequency (the weighting's fmass) of length-`length`
    words for guessed `letters`. Rows: spelled-only-from, uniquely-determined, then one
    row per # of further misses the strategy makes after force-guessing them
    (`misses after = k`; k=0 is the miss-free case). See markdown for semantics."""
    s = sview.make_strategy(strategy, length, weighting=weighting, budget=budget,
                            fill_capped=fill_capped)
    if not s.available:
        print('not available:', s.note)
        return pd.DataFrame()
    freq = s.word_freq()
    rows = _letter_rows(*_letter_stats(s, letters))
    tot_n, tot_f = s.g.N, float(freq.sum())
    df = pd.DataFrame([{'category': name, 'n_words': int(m.sum()),
                        'total_freq': round(float(freq[m].sum()), 1),
                        'pct_words': round(100 * int(m.sum()) / tot_n, 2),
                        'pct_freq': round(100 * float(freq[m].sum()) / tot_f, 2) if tot_f else 0.0}
                       for name, m in rows])
    shown = ''.join(sorted({c for c in letters.lower() if c.isalpha()}))
    print(f"letters = {shown!r}  |  length {length}  |  {s.label}")
    print(f"totals: {tot_n:,} words, freq {tot_f:,.1f}")
    return df

display(letters_summary(GUESSED_LETTERS))

## Summary for a set of guessed letters (all lengths)

`letters_summary_all_lengths(letters)` is the same summary (same rows, including one
per `misses after = k`) swept over **every word length**. Word counts (`n_words`) sum
across lengths; the frequency column uses the **cross-length metric** `freq_words`
(each word contributes `fmass / total_freq_L * N_L`, as in the guess tables), so the
lengths sit on one "effective words" scale and add up sensibly. Under `uniform`,
`freq_words == n_words`. Both `pct_words` and `pct_freq` are shares of the total over
all lengths (the total `freq_words` equals the total word count by construction). The
`misses after = k` rows span `k = 0` up to the max across all lengths. Lengths with no
committed solve are skipped.

In [ ]:
def letters_summary_all_lengths(letters, lengths=range(1, 21), *,
                                strategy=STRATEGY, weighting=WEIGHTING,
                                budget=BUDGET, fill_capped=FILL_CAPPED):
    """`letters_summary` aggregated over every word length. Same rows (per # of
    further misses); `n_words` sums the raw counts and `freq_words` sums the
    cross-length metric fmass/total_freq_L * N_L (== n_words under uniform). Lengths
    with no committed solve are skipped."""
    per, tot_n, used, kmax, any_unsolved = [], 0, [], -1, False
    for L in lengths:
        s = sview.make_strategy(strategy, L, weighting=weighting, budget=budget,
                                fill_capped=fill_capped)
        if not s.available:
            continue
        used.append(L)
        freq = s.word_freq()
        total = float(freq.sum())
        scale = (s.g.N / total) if total else 0.0    # freq share -> effective words
        tot_n += s.g.N
        only_L, uniq, misses_after = _letter_stats(s, letters)
        solved = misses_after >= 0
        if solved.any():
            kmax = max(kmax, int(misses_after[solved].max()))
        any_unsolved = any_unsolved or bool((~solved).any())
        per.append((freq, scale, only_L, uniq, misses_after))
    if not used:
        print(f'no committed solve at any length  [{strategy} / {weighting}]')
        return pd.DataFrame()

    labels = (['only these letters', 'uniquely determined after']
              + [f'misses after = {k}' for k in range(kmax + 1)]
              + (['unsolved (capped/lost)'] if any_unsolved else []))
    n_words = {lab: 0 for lab in labels}
    freq_words = {lab: 0.0 for lab in labels}
    def add(lab, mask, freq, scale):
        n_words[lab] += int(mask.sum())
        freq_words[lab] += float(freq[mask].sum()) * scale
    for freq, scale, only_L, uniq, misses_after in per:
        add('only these letters', only_L, freq, scale)
        add('uniquely determined after', uniq, freq, scale)
        for k in range(kmax + 1):
            add(f'misses after = {k}', misses_after == k, freq, scale)
        if any_unsolved:
            add('unsolved (capped/lost)', misses_after < 0, freq, scale)

    df = pd.DataFrame([{'category': lab, 'n_words': n_words[lab],
                        'freq_words': round(freq_words[lab], 1),
                        'pct_words': round(100 * n_words[lab] / tot_n, 2),
                        'pct_freq': round(100 * freq_words[lab] / tot_n, 2)}  # total freq_words == tot_n
                       for lab in labels])
    shown = ''.join(sorted({c for c in letters.lower() if c.isalpha()}))
    print(f"letters = {shown!r}  |  lengths {used[0]}-{used[-1]} ({len(used)} solved)  |  {strategy} / {weighting}")
    print(f"totals: {tot_n:,} words (= total freq_words under the cross-length normalization)")
    return df

display(letters_summary_all_lengths(GUESSED_LETTERS))

## Letter frequency by position (all lengths)

`letter_position_table(letters)` — for a letter or **list of letters** (`'e'`, or
`'aeiou'`, or `['a','e']`), one row per **(length, position)** giving the proportion
of words, and of frequency (under `WEIGHTING`), whose character at that position is
**any** of `letters`. Positions are **1-indexed** (1 = first letter); proportions are
per-length (the denominator is that length's word / frequency total). Wordlist-only
(independent of `STRATEGY`); sorted by length then position. Under `uniform`,
`prop_freq == prop_words`.

In [ ]:
POS_LETTERS = 'y'   # <-- one or more letters (e.g. 'aeiou'); counts positions holding ANY of them

def letter_position_table(letters, lengths=range(1, 21), *, weighting=WEIGHTING):
    """One row per (length, position): the proportion of length-`length` words -- and
    of their frequency (the weighting's fmass) -- whose character at that position is
    ANY of `letters` (a string like 'aeiou' or a list). Positions are 1-indexed
    (1 = first letter). Wordlist-only (independent of STRATEGY); sorted by length then
    position."""
    Lset = {c for c in ''.join(letters).lower() if c.isalpha()}
    Lidx = [ord(c) - 97 for c in Lset]
    rows = []
    for L in lengths:
        s = sview.make_strategy('greedy', L, weighting=weighting)
        if not s.available:
            continue
        g, freq = s.g, s.word_freq()
        totf = float(freq.sum())
        combo = np.zeros(g.N, dtype=g.masks.dtype)     # per-word: positions holding any listed letter
        for ci in Lidx:
            combo |= g.masks[:, ci]
        for p in range(L):
            has = ((combo >> np.uint32(p)) & np.uint32(1)).astype(bool)
            nw = int(has.sum())
            rows.append({'length': L, 'position': p + 1, 'n_words': nw,
                         'prop_words': round(nw / g.N, 4) if g.N else 0.0,
                         'prop_freq': round(float(freq[has].sum()) / totf, 4) if totf else 0.0})
    df = pd.DataFrame(rows).sort_values(['length', 'position']).reset_index(drop=True)
    print(f"letters {''.join(sorted(Lset))!r}  |  weighting {weighting}  |  {len(df)} (length, position) rows")
    return df

with pd.option_context('display.max_rows', None):
    display(letter_position_table(POS_LETTERS).head(55))

## Most common n-grams containing a letter (all lengths)

`letter_ngram_table(letter)` — the **bigrams, trigrams and (optionally) 4-grams** (adjacent runs) that
contain `letter`, ranked together by the **number of words** (pooled over all lengths) that
contain the n-gram; a word containing it more than once counts **once**. `pct_freq` is
the same but frequency-weighted (cross-length `fmass / total_freq_L * N_L`; `==
pct_words` under `uniform`). The `n` column is the n-gram length (2, 3, or 4). Set `INCLUDE_TRIGRAMS = False`
for bigrams only, or `INCLUDE_FOURGRAMS = True` to also mix in 4-grams. Sorted by word count descending,
**ties broken toward the shorter n-gram**; wordlist-only; uses the `GRAM_LETTER` knob. (Trigrams are rarer than bigrams, so they
naturally sit lower in the ranking. Length-1 words, with no n-gram, are still in the
denominator — it's "% of all words".)

In [ ]:
GRAM_LETTER      = 'y'    # <-- letter the n-grams must contain
INCLUDE_TRIGRAMS = True   # <-- mix trigrams in with the bigrams
INCLUDE_FOURGRAMS = False  # <-- also mix in 4-grams

def letter_ngram_table(letter, lengths=range(1, 21), *, weighting=WEIGHTING,
                       include_trigrams=True, include_fourgrams=False):
    """N-grams (adjacent runs) containing `letter`, ranked by the % of words across
    all lengths that contain the n-gram -- a word with it more than once counts ONCE
    -- with the % of frequency-weighted words (cross-length fmass/total_freq_L * N_L;
    == % of words under uniform). Bigrams, plus trigrams (`include_trigrams`) and
    4-grams (`include_fourgrams`), ranked together. Sorted by word count descending,
    ties broken toward the SHORTER n-gram. `n` is the n-gram length. Wordlist-only
    (independent of STRATEGY)."""
    letter = letter.lower()
    sizes = [2] + ([3] if include_trigrams else []) + ([4] if include_fourgrams else [])
    nwords, fwords, tot_n = {}, {}, 0
    for L in lengths:
        s = sview.make_strategy('greedy', L, weighting=weighting)
        if not s.available:
            continue
        g, freq = s.g, s.word_freq()
        total = float(freq.sum())
        scale = (g.N / total) if total else 0.0       # cross-length effective words
        tot_n += g.N
        for i, w in enumerate(g.words):
            fwi = freq[i] * scale
            seen = {w[j:j + k] for k in sizes for j in range(len(w) - k + 1)
                    if letter in w[j:j + k]}
            for gram in seen:
                nwords[gram] = nwords.get(gram, 0) + 1
                fwords[gram] = fwords.get(gram, 0.0) + fwi
    if not nwords:
        print(f'no n-grams containing {letter!r}')
        return pd.DataFrame(columns=['ngram', 'n', 'n_words', 'pct_words', 'pct_freq'])
    df = pd.DataFrame([{'ngram': gram, 'n': len(gram), 'n_words': nwords[gram],
                        'pct_words': round(100 * nwords[gram] / tot_n, 3),
                        'pct_freq': round(100 * fwords[gram] / tot_n, 3)}
                       for gram in nwords]).sort_values(['n_words', 'n'], ascending=[False, True]).reset_index(drop=True)
    kinds = 'n=' + ','.join(str(k) for k in sizes)
    print(f"letter {letter!r}  |  {kinds}  |  weighting {weighting}  |  {tot_n:,} words, {len(df)} n-grams")
    return df

display(letter_ngram_table(GRAM_LETTER, include_trigrams=INCLUDE_TRIGRAMS, include_fourgrams=INCLUDE_FOURGRAMS).head(50))